In [14]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler

# Step 1: Load the dataset
df = pd.read_csv('insurance_claims.csv')

# Clean column names to remove any unwanted spaces
df.columns = df.columns.str.strip()

# Step 2: Encode categorical variables (Claim_Type and Occupation)
df = pd.get_dummies(df, columns=['Claim_Type', 'Occupation'], drop_first=True)

# Step 3: Split the data into features (X) and target (y)
X = df.drop(columns=['Claim_Status', 'Claim_ID'])  # Drop target and ID columns
y = df['Claim_Status']  # The target variable (Claim Status)

# Step 4: Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Step 5: Standardize the data
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Step 6: Train the Random Forest model
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train_scaled, y_train)

# Step 7: Evaluate the model accuracy
accuracy = model.score(X_test_scaled, y_test)
print(f"Model Accuracy: {accuracy:.2f}")

# Step 8: Function to predict new claims
def predict_claim_approval(new_claim_data):
    # Encode new claim data (same way as training data)
    new_claim_data_encoded = pd.get_dummies(new_claim_data, columns=['Claim_Type', 'Occupation'], drop_first=True)
    
    # Align columns with the training data (fill missing columns with zeros)
    new_claim_data_encoded = new_claim_data_encoded.reindex(columns=X.columns, fill_value=0)
    
    # Scale the new claim data
    new_claim_data_scaled = scaler.transform(new_claim_data_encoded)
    
    # Predict claim approval
    prediction = model.predict(new_claim_data_scaled)
    return "Approved" if prediction == 1 else "Denied"

# Example new claim data
new_claim = pd.DataFrame({
    'Claim_Amount': [10000],
    'Policyholder_Age': [30],
    'Previous_Claim_History': [0],
    'Claim_Type': ['Medical'],  # Note: Raw input data (categorical)
    'Occupation': ['Employed']  # Note: Raw input data (categorical)
})

# Predict the claim approval
prediction = predict_claim_approval(new_claim)
print(f"Claim Status: {prediction}")


Model Accuracy: 0.60
Claim Status: Approved
